# Exercise 03 — Wikipedia Vote Network

Centrality and structural analysis of the Wikipedia admin-election vote network.

In [21]:
import networkx as nx
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
import pandas as pd
from pathlib import Path
from pyvis.network import Network

## Load Data

**Citation:** J. Leskovec, D. Huttenlocher, and J. Kleinberg. Signed networks in social media. CHI 2010.

In [22]:
# Load graph from data file
data_path = Path('data/wiki-Vote.txt')
G = nx.DiGraph()

with open(data_path, 'r') as f:
    for line in f:
        line = line.strip()
        if line.startswith('#'):
            continue
        if not line:
            continue
        parts = line.split('\t')
        if len(parts) >= 2:
            u, v = int(parts[0]), int(parts[1])
            G.add_edge(u, v)

print(f"Graph loaded: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

Graph loaded: 7115 nodes, 103689 edges


## Degree-Based Measures

In [23]:
# In-degree and out-degree statistics
in_degrees = [G.in_degree(n) for n in G.nodes()]
out_degrees = [G.out_degree(n) for n in G.nodes()]

in_deg_cent = nx.in_degree_centrality(G)
out_deg_cent = nx.out_degree_centrality(G)

print("## Degree Statistics")
print()
print("**In-Degree:**")
print("| Metric | Value |")
print("|--------|-------|")
print(f"| Min | {min(in_degrees)} |")
print(f"| Max | {max(in_degrees)} |")
print(f"| Mean | {np.mean(in_degrees):.2f} |")
print(f"| Median | {np.median(in_degrees):.2f} |")
print(f"| Std Dev | {np.std(in_degrees):.2f} |")
print()
print("**Out-Degree:**")
print("| Metric | Value |")
print("|--------|-------|")
print(f"| Min | {min(out_degrees)} |")
print(f"| Max | {max(out_degrees)} |")
print(f"| Mean | {np.mean(out_degrees):.2f} |")
print(f"| Median | {np.median(out_degrees):.2f} |")
print(f"| Std Dev | {np.std(out_degrees):.2f} |")
print()
print("**Top 5 In-Degree Nodes:**")

top_in = sorted(G.in_degree(), key=lambda x: x[1], reverse=True)[:5]
for rank, (node, degree) in enumerate(top_in, 1):
    print(f"  {rank}. Node {node}: {degree}")

print()
print("**Top 5 Out-Degree Nodes:**")
top_out = sorted(G.out_degree(), key=lambda x: x[1], reverse=True)[:5]
for rank, (node, degree) in enumerate(top_out, 1):
    print(f"  {rank}. Node {node}: {degree}")

## Degree Statistics

**In-Degree:**
| Metric | Value |
|--------|-------|
| Min | 0 |
| Max | 457 |
| Mean | 14.57 |
| Median | 0.00 |
| Std Dev | 31.73 |

**Out-Degree:**
| Metric | Value |
|--------|-------|
| Min | 0 |
| Max | 893 |
| Mean | 14.57 |
| Median | 2.00 |
| Std Dev | 42.28 |

**Top 5 In-Degree Nodes:**
  1. Node 4037: 457
  2. Node 15: 361
  3. Node 2398: 340
  4. Node 2625: 331
  5. Node 1297: 309

**Top 5 Out-Degree Nodes:**
  1. Node 2565: 893
  2. Node 766: 773
  3. Node 11: 743
  4. Node 457: 732
  5. Node 2688: 618


## Centrality Measures

In [24]:
# Degree centrality (normalized)
in_deg_cent = nx.in_degree_centrality(G)
out_deg_cent = nx.out_degree_centrality(G)

# Betweenness — sampled (k=500) for performance on 7k-node graph
betweenness = nx.betweenness_centrality(G, k=500, seed=42)

# PageRank
pagerank = nx.pagerank(G, alpha=0.85)

# Eigenvector centrality (numpy version handles convergence on directed graphs)
try:
    eigenvector = nx.eigenvector_centrality_numpy(G)
except Exception:
    eigenvector = {}

# Closeness — computed on largest WCC undirected projection
largest_wcc_nodes = max(nx.weakly_connected_components(G), key=len)
G_wcc = G.subgraph(largest_wcc_nodes)
closeness = nx.closeness_centrality(G_wcc.to_undirected())

# Print top-5 for each measure
print("## Centrality Measures (Top 5 Nodes Each)")
print()
print("**In-Degree Centrality:**")
top_in_deg = sorted(in_deg_cent.items(), key=lambda x: x[1], reverse=True)[:5]
for rank, (node, cent) in enumerate(top_in_deg, 1):
    print(f"  {rank}. Node {node}: {cent:.6f}")

print()
print("**Out-Degree Centrality:**")
top_out_deg = sorted(out_deg_cent.items(), key=lambda x: x[1], reverse=True)[:5]
for rank, (node, cent) in enumerate(top_out_deg, 1):
    print(f"  {rank}. Node {node}: {cent:.6f}")

print()
print("**Betweenness Centrality (sampled, k=500):**")
top_between = sorted(betweenness.items(), key=lambda x: x[1], reverse=True)[:5]
for rank, (node, cent) in enumerate(top_between, 1):
    print(f"  {rank}. Node {node}: {cent:.6f}")

print()
print("**PageRank:**")
top_pr = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:5]
for rank, (node, cent) in enumerate(top_pr, 1):
    print(f"  {rank}. Node {node}: {cent:.6f}")

if eigenvector:
    print()
    print("**Eigenvector Centrality:**")
    top_eigen = sorted(eigenvector.items(), key=lambda x: x[1], reverse=True)[:5]
    for rank, (node, cent) in enumerate(top_eigen, 1):
        print(f"  {rank}. Node {node}: {cent:.6f}")
else:
    print()
    print("**Eigenvector Centrality:** Could not compute")

print()
print("**Closeness Centrality (on largest WCC):**")
top_close = sorted(closeness.items(), key=lambda x: x[1], reverse=True)[:5]
for rank, (node, cent) in enumerate(top_close, 1):
    print(f"  {rank}. Node {node}: {cent:.6f}")

## Centrality Measures (Top 5 Nodes Each)

**In-Degree Centrality:**
  1. Node 4037: 0.064240
  2. Node 15: 0.050745
  3. Node 2398: 0.047793
  4. Node 2625: 0.046528
  5. Node 1297: 0.043435

**Out-Degree Centrality:**
  1. Node 2565: 0.125527
  2. Node 766: 0.108659
  3. Node 11: 0.104442
  4. Node 457: 0.102896
  5. Node 2688: 0.086871

**Betweenness Centrality (sampled, k=500):**
  1. Node 1549: 0.017811
  2. Node 2565: 0.015890
  3. Node 15: 0.011466
  4. Node 72: 0.007503
  5. Node 28: 0.006336

**PageRank:**
  1. Node 4037: 0.004613
  2. Node 15: 0.003681
  3. Node 6634: 0.003525
  4. Node 2625: 0.003286
  5. Node 2398: 0.002605

**Eigenvector Centrality:**
  1. Node 2398: 0.117197
  2. Node 4037: 0.108969
  3. Node 15: 0.098180
  4. Node 4191: 0.095686
  5. Node 2625: 0.095493

**Closeness Centrality (on largest WCC):**
  1. Node 2565: 0.490795
  2. Node 766: 0.470154
  3. Node 457: 0.469841
  4. Node 1549: 0.469092
  5. Node 1166: 0.468906


## Structural Metrics

In [25]:
# Density
density = nx.density(G)

# Average clustering coefficient
avg_clustering = nx.average_clustering(G.to_undirected())

# Path metric on largest SCC (state choice clearly)
largest_scc_nodes = max(nx.strongly_connected_components(G), key=len)
G_scc = G.subgraph(largest_scc_nodes).copy()

# Use eccentricity on a sample of 100 nodes (diameter too slow on 1300-node SCC)
sample_nodes = list(G_scc.nodes())[:100]
ecc = nx.eccentricity(G_scc, v=sample_nodes)
approx_diameter = max(ecc.values()) if ecc else 0

print("## Structural Metrics")
print()
print("| Metric | Value |")
print("|--------|-------|")
print(f"| Density | {density:.6f} |")
print(f"| Average Clustering Coefficient | {avg_clustering:.6f} |")
print(f"| Approximate Diameter | {approx_diameter} |")
print()
print(f"**Note:** Approximate diameter computed on largest SCC of {len(largest_scc_nodes)} nodes, eccentricity sampled from {len(sample_nodes)} nodes.")

## Structural Metrics

| Metric | Value |
|--------|-------|
| Density | 0.002049 |
| Average Clustering Coefficient | 0.140898 |
| Approximate Diameter | 7 |

**Note:** Approximate diameter computed on largest SCC of 1300 nodes, eccentricity sampled from 100 nodes.


## Centrality Visualization

In [27]:
# Build subgraph of top-100 PageRank nodes + their mutual edges
top_100_pr_nodes = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)[:100]
top_100_nodes = set([node for node, _ in top_100_pr_nodes])

# Create subgraph with only edges between top 100 nodes
G_top = G.subgraph(top_100_nodes).copy()

# Create PyVis network
net = Network(directed=True, height='750px', width='100%')
net.from_nx(G_top)

# Disable physics for stable visualization
net.toggle_physics(False)

# Find top PageRank node for coloring
top_pr_node = max(pagerank.items(), key=lambda x: x[1])[0]

# Scale PageRank for node sizes and colors
pr_values = [pagerank.get(n, 0) for n in G_top.nodes()]
pr_min, pr_max = min(pr_values), max(pr_values)

# Precompute PageRank ranking for color gradient
pr_ranked = sorted(pagerank.items(), key=lambda x: x[1], reverse=True)
pr_rank_dict = {node: i for i, (node, _) in enumerate(pr_ranked)}

# Update node visualization properties
for node in G_top.nodes():
    pr_value = pagerank.get(node, 0)
    
    # Scale size proportional to PageRank
    size = 10 + 50 * (pr_value - pr_min) / (pr_max - pr_min) if pr_max > pr_min else 10
    
    # Color gradient: red for top node, lightblue to orange for others
    if node == top_pr_node:
        color = 'red'
    else:
        rank = pr_rank_dict.get(node, 100)
        ratio = min(rank / 100.0, 1.0)
        # Interpolate from lightblue (135, 206, 235) to orange (255, 165, 0)
        r = int(135 + (255 - 135) * ratio)
        g = int(206 + (165 - 206) * ratio)
        b = int(235 + (0 - 235) * ratio)
        color = f'rgb({r}, {g}, {b})'
    
    net.get_node(node)['size'] = size
    net.get_node(node)['color'] = color
    net.get_node(node)['title'] = f"Node {node}: PR={pr_value:.6f}"

# Write HTML file with stable layout
net.write_html('centrality_viz.html', notebook=False)
print("Centrality visualization saved as centrality_viz.html")

Centrality visualization saved as centrality_viz.html


## Comparison: In-Degree vs Out-Degree vs PageRank

Direction fundamentally changes importance. In-degree represents received trust (voted for), out-degree represents active participation (who votes), and PageRank represents authority propagated through chains of recommendations. These often identify very different sets of nodes.

In [28]:
# Build comparison table
top_n = 5
in_top = sorted(in_deg_cent, key=in_deg_cent.get, reverse=True)[:top_n]
out_top = sorted(out_deg_cent, key=out_deg_cent.get, reverse=True)[:top_n]
pr_top = sorted(pagerank, key=pagerank.get, reverse=True)[:top_n]

print("## Top 5 Nodes by Metric")
print()
print("| Rank | In-Degree | Out-Degree | PageRank |")
print("|------|-----------|------------|----------|")
for i in range(top_n):
    print(f"| {i+1} | Node {in_top[i]} | Node {out_top[i]} | Node {pr_top[i]} |")

## Top 5 Nodes by Metric

| Rank | In-Degree | Out-Degree | PageRank |
|------|-----------|------------|----------|
| 1 | Node 4037 | Node 2565 | Node 4037 |
| 2 | Node 15 | Node 766 | Node 15 |
| 3 | Node 2398 | Node 11 | Node 6634 |
| 4 | Node 2625 | Node 457 | Node 2625 |
| 5 | Node 1297 | Node 2688 | Node 2398 |


## Conclusion & Interpretation

In [29]:
# Interpretation
print("## Interpretation: How Direction Changes Importance")
print()
print("### Most Trusted Candidates (In-Degree)")
print(f"The nodes with highest in-degree are those who received the most votes. These candidates represent widely-trusted members of the Wikipedia community. The top in-degree nodes are: {', '.join(map(str, in_top))}")
print()
print("### Most Active Voters (Out-Degree)")
print(f"The nodes with highest out-degree are those who cast the most votes. These represent highly engaged community members who participate actively in admin elections. Notably, the top out-degree nodes ({', '.join(map(str, out_top))}) often differ significantly from top in-degree nodes, reflecting different roles: being trusted vs. being an active participant.")
print()
print("### Propagated Authority (PageRank)")
print(f"PageRank nodes ({', '.join(map(str, pr_top))}) reflect authority that propagates through chains of recommendations. These are often closer to in-degree rankings than out-degree, as PageRank accumulates incoming authority. A node with high PageRank is trusted by other highly-trusted nodes.")
print()
print("### Why Other Centralities Differ")
print("Betweenness centrality identifies bridge nodes in voting chains—nodes that lie on many shortest paths between others. Eigenvector centrality emphasizes recursive prestige: being voted for by highly-esteemed voters. These differ from PageRank and degree measures because they capture structural position rather than direct voting counts.")
print()
print("### Key Takeaway")
print("Direction fundamentally changes who appears 'important.' A node with high in-degree is popular; high out-degree means highly active; high PageRank means trusted by the trusted. These are distinct network roles, and understanding a node's importance requires knowing which measure is relevant to your question.")

## Interpretation: How Direction Changes Importance

### Most Trusted Candidates (In-Degree)
The nodes with highest in-degree are those who received the most votes. These candidates represent widely-trusted members of the Wikipedia community. The top in-degree nodes are: 4037, 15, 2398, 2625, 1297

### Most Active Voters (Out-Degree)
The nodes with highest out-degree are those who cast the most votes. These represent highly engaged community members who participate actively in admin elections. Notably, the top out-degree nodes (2565, 766, 11, 457, 2688) often differ significantly from top in-degree nodes, reflecting different roles: being trusted vs. being an active participant.

### Propagated Authority (PageRank)
PageRank nodes (4037, 15, 6634, 2625, 2398) reflect authority that propagates through chains of recommendations. These are often closer to in-degree rankings than out-degree, as PageRank accumulates incoming authority. A node with high PageRank is trusted by other highly-tru